# Telepathy on a free Kaggle GPU
Settings: **Accelerator = GPU T4 x2 or P100**, **Internet = On**. T4/P100 have no bfloat16, so everything uses float16.
Add the dataset `telepathy` (the zip) via *Add Input*.

**Step 1 is the gate:** 20 GSM8K questions, text vs latent. PASS = at least 1.5x faster and accuracy within 5 points.

In [ ]:
!pip install -q transformers==5.19.0 datasets==5.1.0 openai python-dotenv httpx
import torch; print(torch.__version__, torch.cuda.get_device_name(0))

In [ ]:
import glob, os, shutil
hit = glob.glob('/kaggle/input/**/bench/run_bench.py', recursive=True)[0]
src = os.path.dirname(os.path.dirname(hit))
shutil.copytree(src, '/kaggle/working/repo', dirs_exist_ok=True)
%cd /kaggle/working/repo
!ls

In [ ]:
# Only if the model download asks you to log in. Paste your own token when prompted; never save it in the notebook.
# from huggingface_hub import login; login()

In [ ]:
# STEP 1: THE GATE (resumable: if the session stops, run this cell again)
!python -m bench.run_bench --task gsm8k --split all --n 20 --modes text,latent --steps 10 --dtype float16 --device cuda --out bench/results/gate
!python -m bench.gate bench/results/gate

In [ ]:
# Only if the gate said NOT YET: try other latent steps
!python -m bench.run_bench --task gsm8k --split all --n 20 --modes latent --steps 5,20 --dtype float16 --device cuda --out bench/results/gate
!python -m bench.gate bench/results/gate

In [ ]:
# STEP 2 (only after PASS): dev split sweep to tune latent steps
!python -m bench.run_bench --task gsm8k --split dev --modes single,text,latent --steps 0,5,10,20 --dtype float16 --device cuda --out bench/results/dev
!python -m bench.metrics bench/results/dev

In [ ]:
# STEP 3: report split
!python -m bench.run_bench --task gsm8k --split test --modes single,text,latent --steps 10 --dtype float16 --device cuda --out bench/results/test
!python -m bench.metrics bench/results/test

In [ ]:
# STEP 4: HONESTY BENCHMARK (resumable). ~30 questions, a few minutes each batch; re-run the cell if the session stops
!python -m bench.honesty_bench --task gsm8k --split dev --n 30 --steps 10 --dtype float16 --device cuda --out bench/results/honesty
!python -m bench.metrics bench/results/honesty

In [ ]:
# Save results so you can download them from the Output tab
!mkdir -p /kaggle/working/out && cp -r bench/results/* /kaggle/working/out/